# 🎓 Emotion Recognition for Student Engagement Detection
## FER2013 Dataset – Deep Learning Pipeline

**Goal:** Train a robust CNN-based emotion classifier on FER2013 that can later serve as the
foundation for student productivity and engagement analysis in classrooms.

### Pipeline Overview
1. **Setup & Imports**
2. **Dataset Extraction & EDA**
3. **Preprocessing & Augmentation**
4. **Baseline Custom CNN**
5. **Transfer Learning with MobileNetV2**
6. **Transfer Learning with EfficientNetB0**
7. **Model Evaluation & Comparison**
8. **Confusion Matrix & Classification Report**
9. **Saving Outputs**
10. **Classroom Engagement Adaptation Notes**

---
**Dataset structure expected:**
```
archive.zip
├── train/
│   ├── angry/ | disgust/ | fear/ | happy/ | neutral/ | sad/ | surprise/
└── test/
    ├── angry/ | disgust/ | fear/ | happy/ | neutral/ | sad/ | surprise/
```

## 1. 📦 Setup & Imports

In [ ]:
# ── Install dependencies (run once in Kaggle/Colab/local) ──────────────────
# !pip install tensorflow scikit-learn matplotlib seaborn opencv-python -q

import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense,
    Dropout, BatchNormalization, Input, Flatten
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, TensorBoard
)
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    precision_score, recall_score
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Global config ──────────────────────────────────────────────────────────
IMG_SIZE       = 48       # FER2013 native resolution
IMG_SIZE_TL    = 96       # Larger size for transfer learning models
BATCH_SIZE     = 64
EPOCHS_BASELINE = 60
EPOCHS_TL       = 40
EMOTIONS       = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
NUM_CLASSES    = len(EMOTIONS)
OUTPUT_DIR     = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU available      : {len(tf.config.list_physical_devices("GPU")) > 0}')
print(f'Emotion classes    : {EMOTIONS}')

## 2. 📂 Dataset Extraction & Exploratory Data Analysis

In [ ]:
# ── Dataset paths ─────────────────────────────────────────────────────────
#
#  The notebook expects this folder layout (same directory as the notebook):
#
#  emotion rec/
#  ├── emotion_recognition.ipynb   ← this file
#  ├── train/
#  │   ├── angry/ disgust/ fear/ happy/ neutral/ sad/ surprise/
#  └── test/
#      ├── angry/ disgust/ fear/ happy/ neutral/ sad/ surprise/
#
#  If your train/ and test/ folders sit elsewhere, update the two
#  variables below accordingly.
# ──────────────────────────────────────────────────────────────────────────

import os

# ✏️  Change these paths if your folder structure is different
TRAIN_DIR = 'train'
TEST_DIR  = 'test'

# ── Validate that the folders exist and contain emotion subfolders ─────────
def validate_split_dir(path, split_name):
    assert os.path.isdir(path), (
        f"Cannot find {split_name}/ folder at '{path}'.
"
        f"Make sure the notebook is in the same folder as train/ and test/,
"
        f"or update TRAIN_DIR / TEST_DIR above."
    )
    found = [e for e in EMOTIONS if os.path.isdir(os.path.join(path, e))]
    assert found, f"No emotion subfolders found inside {path}/"
    missing = [e for e in EMOTIONS if e not in found]
    if missing:
        print(f"  ⚠️  Missing emotion folders in {split_name}/: {missing}")
    return path

TRAIN_DIR = validate_split_dir(TRAIN_DIR, 'train')
TEST_DIR  = validate_split_dir(TEST_DIR,  'test')

print(f'✅ Train dir : {os.path.abspath(TRAIN_DIR)}')
print(f'✅ Test  dir : {os.path.abspath(TEST_DIR)}')


In [ ]:
# ── Count images per class ─────────────────────────────────────────────────
def count_images(split_dir):
    counts = {}
    for emotion in EMOTIONS:
        path = os.path.join(split_dir, emotion)
        if os.path.exists(path):
            counts[emotion] = len([
                f for f in os.listdir(path)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))
            ])
        else:
            counts[emotion] = 0
    return counts

train_counts = count_images(TRAIN_DIR)
test_counts  = count_images(TEST_DIR)

df_counts = pd.DataFrame({
    'Emotion': EMOTIONS,
    'Train'  : [train_counts[e] for e in EMOTIONS],
    'Test'   : [test_counts[e]  for e in EMOTIONS],
})
df_counts['Total'] = df_counts['Train'] + df_counts['Test']
df_counts = df_counts.sort_values('Train', ascending=False).reset_index(drop=True)

print('\n📊 Image distribution:')
print(df_counts.to_string(index=False))
print(f'\nTotal train images : {df_counts["Train"].sum():,}')
print(f'Total test  images : {df_counts["Test"].sum():,}')

# ── Imbalance ratio ────────────────────────────────────────────────────────
max_c = df_counts['Train'].max()
min_c = df_counts['Train'].min()
print(f'\nClass imbalance ratio (max/min train): {max_c/min_c:.1f}x')
print('→ Class weighting will be applied during training.')

In [ ]:
# ── Visualise class distribution ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.Set2(np.linspace(0, 1, NUM_CLASSES))

for ax, split, col in zip(axes, ['Train', 'Test'], [colors, colors]):
    vals   = df_counts[split].values
    labels = df_counts['Emotion'].values
    bars   = ax.bar(labels, vals, color=col, edgecolor='black', linewidth=0.5)
    ax.set_title(f'{split} Set – Class Distribution', fontsize=13, fontweight='bold')
    ax.set_xlabel('Emotion')
    ax.set_ylabel('Number of Images')
    ax.tick_params(axis='x', rotation=30)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                str(v), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'class_distribution.png'), dpi=150)
plt.show()
print('Saved class_distribution.png')

In [ ]:
# ── Sample images from each class ─────────────────────────────────────────
fig, axes = plt.subplots(NUM_CLASSES, 5, figsize=(12, 2.5*NUM_CLASSES))
fig.suptitle('Sample Images per Emotion Class (Train Set)', fontsize=14, fontweight='bold', y=1.01)

for row, emotion in enumerate(EMOTIONS):
    emotion_dir = os.path.join(TRAIN_DIR, emotion)
    imgs = random.sample(os.listdir(emotion_dir), min(5, len(os.listdir(emotion_dir))))
    for col, img_name in enumerate(imgs):
        img = cv2.imread(os.path.join(emotion_dir, img_name), cv2.IMREAD_GRAYSCALE)
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(emotion.capitalize(), fontsize=11,
                                       rotation=0, labelpad=55, va='center')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sample_images.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3. 🔧 Preprocessing & Data Augmentation

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
#  ImageDataGenerators
#
#  • Train: heavy augmentation to improve generalisation and combat
#    the small, imbalanced FER2013 training set.
#  • Validation/Test: only rescaling (no random transforms).
# ─────────────────────────────────────────────────────────────────────────

def build_generators(img_size, batch_size, color_mode='grayscale',
                     validation_split=0.15):
    """
    Build train/val/test ImageDataGenerators.

    Parameters
    ----------
    img_size         : int  – target square size (48 for CNN, 96 for TL)
    batch_size       : int
    color_mode       : 'grayscale' | 'rgb'
    validation_split : float – fraction of train data for validation

    Returns
    -------
    train_gen, val_gen, test_gen
    """
    train_aug = ImageDataGenerator(
        rescale           = 1.0 / 255,
        rotation_range    = 20,        # random rotations ±20°
        width_shift_range = 0.15,      # horizontal shifts
        height_shift_range= 0.15,      # vertical shifts
        shear_range       = 0.10,      # shearing
        zoom_range        = 0.15,      # zoom in/out
        horizontal_flip   = True,      # mirror faces horizontally
        brightness_range  = [0.75, 1.25],  # random brightness
        fill_mode         = 'nearest',
        validation_split  = validation_split,
    )

    test_aug = ImageDataGenerator(rescale=1.0 / 255)

    common_kwargs = dict(
        directory   = TRAIN_DIR,
        target_size = (img_size, img_size),
        batch_size  = batch_size,
        color_mode  = color_mode,
        class_mode  = 'categorical',
        classes     = EMOTIONS,
        seed        = SEED,
    )

    train_gen = train_aug.flow_from_directory(
        **common_kwargs, subset='training', shuffle=True
    )
    val_gen   = train_aug.flow_from_directory(
        **common_kwargs, subset='validation', shuffle=False
    )
    test_gen  = test_aug.flow_from_directory(
        directory   = TEST_DIR,
        target_size = (img_size, img_size),
        batch_size  = batch_size,
        color_mode  = color_mode,
        class_mode  = 'categorical',
        classes     = EMOTIONS,
        shuffle     = False,
    )
    return train_gen, val_gen, test_gen


# ── Grayscale generators for custom CNN (48×48×1) ─────────────────────────
train_gen_gray, val_gen_gray, test_gen_gray = build_generators(
    img_size=IMG_SIZE, batch_size=BATCH_SIZE, color_mode='grayscale'
)

# ── RGB generators for transfer learning models (96×96×3) ─────────────────
train_gen_rgb, val_gen_rgb, test_gen_rgb = build_generators(
    img_size=IMG_SIZE_TL, batch_size=BATCH_SIZE, color_mode='rgb'
)

print(f'Train batches (gray) : {len(train_gen_gray)}')
print(f'Val   batches (gray) : {len(val_gen_gray)}')
print(f'Test  batches (gray) : {len(test_gen_gray)}')

In [ ]:
# ── Compute class weights to handle imbalance ──────────────────────────────
# We compute weights from the actual file-count distribution, not from a
# generator, so they are exact regardless of batch sampling order.

y_train_labels = np.array([
    idx
    for idx, emotion in enumerate(EMOTIONS)
    for _ in range(train_counts[emotion])
])

cw_values = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=y_train_labels
)
class_weights = dict(enumerate(cw_values))

print('Class weights (balancing imbalanced dataset):')
for idx, emotion in enumerate(EMOTIONS):
    print(f'  {emotion:10s} → {class_weights[idx]:.4f}')

In [ ]:
# ── Visualise augmented samples ────────────────────────────────────────────
batch_imgs, batch_labels = next(iter(train_gen_gray))

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle('Augmented Training Samples (grayscale, rescaled to [0,1])',
             fontsize=12, fontweight='bold')
for i, ax in enumerate(axes.flatten()):
    if i < len(batch_imgs):
        ax.imshow(batch_imgs[i].squeeze(), cmap='gray', vmin=0, vmax=1)
        label_idx = np.argmax(batch_labels[i])
        ax.set_title(EMOTIONS[label_idx], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'augmented_samples.png'), dpi=150)
plt.show()

## 4. 🧠 Model 1 – Baseline Custom CNN

A deep custom CNN tuned specifically for the 48×48 grayscale FER2013 images.  
Architecture highlights:
- 4 convolutional blocks with increasing filter depth
- Batch normalisation after every Conv layer
- Spatial Dropout on feature maps
- Dense head with L2 regularisation and standard Dropout

In [ ]:
def build_custom_cnn(input_shape=(48, 48, 1), num_classes=7,
                     l2_lambda=1e-4):
    """
    Custom CNN for FER2013 (48×48 grayscale).

    Block design:
        Conv2D → BN → ReLU → Conv2D → BN → ReLU → MaxPool → SpatialDropout
    Repeated 4 times with doubling filter counts (32 → 64 → 128 → 256).
    """
    reg = regularizers.l2(l2_lambda)
    inp = Input(shape=input_shape, name='input')
    x   = inp

    # ── Convolutional blocks ───────────────────────────────────────────────
    for filters, sp_drop in [(32, 0.1), (64, 0.1), (128, 0.2), (256, 0.2)]:
        x = Conv2D(filters, 3, padding='same', kernel_regularizer=reg)(x)
        x = BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = Conv2D(filters, 3, padding='same', kernel_regularizer=reg)(x)
        x = BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = MaxPooling2D(2)(x)
        x = layers.SpatialDropout2D(sp_drop)(x)

    # ── Classification head ────────────────────────────────────────────────
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation='relu', kernel_regularizer=reg)(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu', kernel_regularizer=reg)(x)
    x = Dropout(0.4)(x)
    out = Dense(num_classes, activation='softmax', name='predictions')(x)

    model = Model(inp, out, name='CustomCNN')
    return model


cnn_model = build_custom_cnn()
cnn_model.summary()

In [ ]:
# ── Compile ────────────────────────────────────────────────────────────────
# Adam with warm-restart-like schedule via ReduceLROnPlateau.
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ── Callbacks ─────────────────────────────────────────────────────────────
cnn_ckpt_path = os.path.join(OUTPUT_DIR, 'best_custom_cnn.h5')

cnn_callbacks = [
    EarlyStopping(
        monitor='val_accuracy', patience=12,
        restore_best_weights=True, verbose=1
    ),
    ModelCheckpoint(
        cnn_ckpt_path, monitor='val_accuracy',
        save_best_only=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=6,
        min_lr=1e-6, verbose=1
    ),
]

print('Callbacks configured:', [cb.__class__.__name__ for cb in cnn_callbacks])

In [ ]:
# ── Train ──────────────────────────────────────────────────────────────────
print('Training Custom CNN …')
cnn_history = cnn_model.fit(
    train_gen_gray,
    validation_data  = val_gen_gray,
    epochs           = EPOCHS_BASELINE,
    callbacks        = cnn_callbacks,
    class_weight     = class_weights,
    verbose          = 1,
)
print(f'\nBest val_accuracy : {max(cnn_history.history["val_accuracy"]):.4f}')

## 5. 🚀 Model 2 – Transfer Learning with MobileNetV2

MobileNetV2 is lightweight and well-suited for real-time classroom inference.  
Strategy: **freeze** the base, train only the head → then **fine-tune** the top layers.

In [ ]:
def build_mobilenet_model(img_size=96, num_classes=7, l2_lambda=1e-4):
    """
    MobileNetV2 transfer learning model.
    Input: RGB images of shape (img_size, img_size, 3)
    """
    base = MobileNetV2(
        input_shape = (img_size, img_size, 3),
        include_top = False,
        weights     = 'imagenet',
    )
    base.trainable = False   # freeze during head training

    reg = regularizers.l2(l2_lambda)
    inp = Input(shape=(img_size, img_size, 3), name='input')

    # MobileNetV2 expects inputs in [-1, 1]; our generator outputs [0, 1]
    x = layers.Lambda(lambda z: z * 2.0 - 1.0, name='rescale_to_mobilenet')(inp)
    x = base(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation='relu', kernel_regularizer=reg)(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu', kernel_regularizer=reg)(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax', name='predictions')(x)

    model = Model(inp, out, name='MobileNetV2_FER')
    return model, base


mobilenet_model, mobilenet_base = build_mobilenet_model(img_size=IMG_SIZE_TL)

mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
mobilenet_model.summary()

In [ ]:
# ── Phase 1: Train head only ───────────────────────────────────────────────
mn_ckpt = os.path.join(OUTPUT_DIR, 'best_mobilenet.h5')
mn_callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint(mn_ckpt, monitor='val_accuracy',
                    save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5,
                      min_lr=1e-7, verbose=1),
]

print('Phase 1 – Training head (base frozen) …')
mn_history_phase1 = mobilenet_model.fit(
    train_gen_rgb,
    validation_data = val_gen_rgb,
    epochs          = 20,
    callbacks       = mn_callbacks,
    class_weight    = class_weights,
    verbose         = 1,
)
print(f'Phase 1 best val_accuracy: {max(mn_history_phase1.history["val_accuracy"]):.4f}')

In [ ]:
# ── Phase 2: Fine-tune top layers of MobileNetV2 ──────────────────────────
# Unfreeze the top 30 layers of the base for fine-tuning.
# Use a lower learning rate to avoid destroying ImageNet features.

mobilenet_base.trainable = True
for layer in mobilenet_base.layers[:-30]:
    layer.trainable = False

mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),   # 100× lower LR
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f'Trainable layers after unfreezing top 30: '
      f'{sum(1 for l in mobilenet_base.layers if l.trainable)}')

print('\nPhase 2 – Fine-tuning top 30 base layers …')
mn_history_phase2 = mobilenet_model.fit(
    train_gen_rgb,
    validation_data = val_gen_rgb,
    epochs          = EPOCHS_TL,
    callbacks       = mn_callbacks,
    class_weight    = class_weights,
    verbose         = 1,
)
print(f'Phase 2 best val_accuracy: {max(mn_history_phase2.history["val_accuracy"]):.4f}')

# Merge histories for plotting
def merge_histories(h1, h2):
    merged = {}
    for k in h1.history:
        merged[k] = h1.history[k] + h2.history[k]
    return merged

mn_history_combined = merge_histories(mn_history_phase1, mn_history_phase2)

## 6. ⚡ Model 3 – Transfer Learning with EfficientNetB0

EfficientNet scales depth/width/resolution jointly for high accuracy per FLOP.
Same two-phase approach: freeze → fine-tune.

In [ ]:
def build_efficientnet_model(img_size=96, num_classes=7, l2_lambda=1e-4):
    """
    EfficientNetB0 transfer learning model.
    EfficientNet's built-in preprocessing expects raw [0, 255] uint8 inputs;
    since our generator rescales to [0, 1] we undo that with a Lambda layer.
    """
    base = EfficientNetB0(
        input_shape = (img_size, img_size, 3),
        include_top = False,
        weights     = 'imagenet',
    )
    base.trainable = False

    reg = regularizers.l2(l2_lambda)
    inp = Input(shape=(img_size, img_size, 3), name='input')

    # EfficientNet includes its own normalisation internally;
    # undo the generator rescale so we pass [0, 255] as expected.
    x = layers.Lambda(lambda z: z * 255.0, name='undo_rescale')(inp)
    x = base(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation='relu', kernel_regularizer=reg)(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu', kernel_regularizer=reg)(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax', name='predictions')(x)

    model = Model(inp, out, name='EfficientNetB0_FER')
    return model, base


effnet_model, effnet_base = build_efficientnet_model(img_size=IMG_SIZE_TL)

effnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
effnet_model.summary()

In [ ]:
# ── Phase 1: Train head only ───────────────────────────────────────────────
en_ckpt = os.path.join(OUTPUT_DIR, 'best_efficientnet.h5')
en_callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint(en_ckpt, monitor='val_accuracy',
                    save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5,
                      min_lr=1e-7, verbose=1),
]

print('Phase 1 – Training head (EfficientNetB0 base frozen) …')
en_history_phase1 = effnet_model.fit(
    train_gen_rgb,
    validation_data = val_gen_rgb,
    epochs          = 20,
    callbacks       = en_callbacks,
    class_weight    = class_weights,
    verbose         = 1,
)
print(f'Phase 1 best val_accuracy: {max(en_history_phase1.history["val_accuracy"]):.4f}')

In [ ]:
# ── Phase 2: Fine-tune top layers ─────────────────────────────────────────
effnet_base.trainable = True
for layer in effnet_base.layers[:-40]:   # keep deeper features frozen
    layer.trainable = False

effnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-6),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('Phase 2 – Fine-tuning top 40 EfficientNetB0 layers …')
en_history_phase2 = effnet_model.fit(
    train_gen_rgb,
    validation_data = val_gen_rgb,
    epochs          = EPOCHS_TL,
    callbacks       = en_callbacks,
    class_weight    = class_weights,
    verbose         = 1,
)
print(f'Phase 2 best val_accuracy: {max(en_history_phase2.history["val_accuracy"]):.4f}')

en_history_combined = merge_histories(en_history_phase1, en_history_phase2)

## 7. 📊 Training Curves & Model Comparison

In [ ]:
def plot_history(history_dict, model_name, save_path=None):
    """
    Plot training & validation accuracy + loss curves.
    Shaded region highlights potential overfitting (train > val by >5%).
    """
    acc     = history_dict['accuracy']
    val_acc = history_dict['val_accuracy']
    loss    = history_dict['loss']
    val_loss= history_dict['val_loss']
    epochs  = range(1, len(acc) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{model_name} – Training Curves', fontsize=13, fontweight='bold')

    # Accuracy
    ax1.plot(epochs, acc,     label='Train acc',  color='royalblue')
    ax1.plot(epochs, val_acc, label='Val acc',    color='darkorange')
    ax1.fill_between(epochs,
                     [max(0, a-v-0.05) for a, v in zip(acc, val_acc)],
                     0, alpha=0.08, color='red', label='Overfit zone')
    ax1.set_title('Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(alpha=0.3)
    ax1.set_ylim(0, 1)

    # Loss
    ax2.plot(epochs, loss,     label='Train loss', color='royalblue')
    ax2.plot(epochs, val_loss, label='Val loss',   color='darkorange')
    ax2.set_title('Cross-Entropy Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        print(f'Saved {save_path}')
    plt.show()


plot_history(cnn_history.history, 'Custom CNN',
             save_path=os.path.join(OUTPUT_DIR, 'history_custom_cnn.png'))

plot_history(mn_history_combined, 'MobileNetV2',
             save_path=os.path.join(OUTPUT_DIR, 'history_mobilenet.png'))

plot_history(en_history_combined, 'EfficientNetB0',
             save_path=os.path.join(OUTPUT_DIR, 'history_efficientnet.png'))

## 8. 🔍 Model Evaluation – Confusion Matrix & Classification Report

In [ ]:
def evaluate_model(model, test_generator, model_name):
    """
    Full evaluation pipeline:
        1. Generate predictions on the test set
        2. Confusion matrix (normalised + raw)
        3. Classification report (precision / recall / F1 per class)
        4. Macro-averaged metrics summary

    Returns
    -------
    dict with test_acc, macro_f1, predictions, true_labels
    """
    print(f'\n{"="*60}')
    print(f' Evaluating: {model_name}')
    print(f'{"="*60}')

    # ── Get predictions ────────────────────────────────────────────────
    test_generator.reset()
    y_prob = model.predict(test_generator, verbose=1)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = test_generator.classes

    # ── Keras test loss & accuracy ─────────────────────────────────────
    test_generator.reset()
    test_loss, test_acc = model.evaluate(test_generator, verbose=0)
    print(f'\nTest loss     : {test_loss:.4f}')
    print(f'Test accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)')

    # ── Sklearn metrics ────────────────────────────────────────────────
    macro_f1  = f1_score(y_true, y_pred, average='macro')
    macro_pre = precision_score(y_true, y_pred, average='macro')
    macro_rec = recall_score(y_true, y_pred, average='macro')
    print(f'Macro F1      : {macro_f1:.4f}')
    print(f'Macro Prec.   : {macro_pre:.4f}')
    print(f'Macro Recall  : {macro_rec:.4f}')

    # ── Classification report ──────────────────────────────────────────
    print(f'\nPer-class report:')
    report = classification_report(y_true, y_pred,
                                    target_names=EMOTIONS, digits=4)
    print(report)

    # ── Normalised confusion matrix ────────────────────────────────────
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'{model_name} – Confusion Matrix', fontsize=13, fontweight='bold')

    for ax, data, fmt, title in [
        (axes[0], cm_norm, '.2%', 'Normalised (row = true class)'),
        (axes[1], cm,      'd',   'Raw counts'),
    ]:
        sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                    xticklabels=EMOTIONS, yticklabels=EMOTIONS,
                    linewidths=0.5, ax=ax)
        ax.set_title(title)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        ax.tick_params(axis='x', rotation=30)

    plt.tight_layout()
    safe_name = model_name.lower().replace(' ', '_')
    cm_path = os.path.join(OUTPUT_DIR, f'confusion_matrix_{safe_name}.png')
    plt.savefig(cm_path, dpi=150)
    print(f'Saved {cm_path}')
    plt.show()

    return {
        'model_name' : model_name,
        'test_acc'   : test_acc,
        'test_loss'  : test_loss,
        'macro_f1'   : macro_f1,
        'macro_pre'  : macro_pre,
        'macro_rec'  : macro_rec,
        'y_pred'     : y_pred,
        'y_true'     : y_true,
        'report'     : report,
    }


# ── Run evaluations ────────────────────────────────────────────────────────
results_cnn     = evaluate_model(cnn_model,      test_gen_gray, 'Custom CNN')
results_mn      = evaluate_model(mobilenet_model, test_gen_rgb,  'MobileNetV2')
results_en      = evaluate_model(effnet_model,    test_gen_rgb,  'EfficientNetB0')

In [ ]:
# ── Side-by-side comparison table ─────────────────────────────────────────
all_results = [results_cnn, results_mn, results_en]

df_compare = pd.DataFrame([
    {
        'Model'       : r['model_name'],
        'Test Acc.'   : f"{r['test_acc']*100:.2f}%",
        'Test Loss'   : f"{r['test_loss']:.4f}",
        'Macro F1'    : f"{r['macro_f1']:.4f}",
        'Macro Prec.' : f"{r['macro_pre']:.4f}",
        'Macro Rec.'  : f"{r['macro_rec']:.4f}",
    }
    for r in all_results
])

print('\n' + '='*70)
print('  MODEL COMPARISON SUMMARY')
print('='*70)
print(df_compare.to_string(index=False))
print('='*70)

# Save comparison
df_compare.to_csv(os.path.join(OUTPUT_DIR, 'model_comparison.csv'), index=False)
print('Saved model_comparison.csv')

In [ ]:
# ── Visual comparison bar chart ────────────────────────────────────────────
metrics   = ['test_acc', 'macro_f1', 'macro_pre', 'macro_rec']
m_labels  = ['Test Accuracy', 'Macro F1', 'Macro Precision', 'Macro Recall']
model_names = [r['model_name'] for r in all_results]

x = np.arange(len(metrics))
width = 0.25
colors = ['#4C72B0', '#DD8452', '#55A868']

fig, ax = plt.subplots(figsize=(12, 6))
for i, (r, color) in enumerate(zip(all_results, colors)):
    vals = [r[m] for m in metrics]
    bars = ax.bar(x + i*width, vals, width, label=r['model_name'],
                  color=color, edgecolor='black', linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(m_labels, fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model_comparison.png'), dpi=150)
plt.show()

In [ ]:
# ── Per-class F1 comparison across models ─────────────────────────────────
def per_class_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average=None, labels=range(NUM_CLASSES))

f1_matrix = np.array([
    per_class_f1(r['y_true'], r['y_pred']) for r in all_results
])  # shape (3, 7)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(NUM_CLASSES)
for i, (name, color) in enumerate(zip(model_names, colors)):
    ax.plot(x, f1_matrix[i], marker='o', label=name, color=color, linewidth=2)

ax.set_xticks(x)
ax.set_xticklabels([e.capitalize() for e in EMOTIONS], fontsize=11)
ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1)
ax.set_title('Per-Class F1 Score by Model', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
ax.axhline(0.5, color='red', linestyle='--', alpha=0.4, label='50% threshold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'per_class_f1.png'), dpi=150)
plt.show()

# Show as table
df_f1 = pd.DataFrame(f1_matrix, columns=EMOTIONS, index=model_names)
df_f1['Mean F1'] = df_f1.mean(axis=1)
print('\nPer-class F1 scores:')
print(df_f1.round(4).to_string())

## 9. 💾 Save Best Model & Outputs

In [ ]:
# ── Determine best model by test accuracy ─────────────────────────────────
best_result = max(all_results, key=lambda r: r['test_acc'])
print(f'🏆 Best model: {best_result["model_name"]} '
      f'(test acc = {best_result["test_acc"]*100:.2f}%)')

# ── Map result to model object & save path ────────────────────────────────
model_map = {
    'Custom CNN'      : (cnn_model,      os.path.join(OUTPUT_DIR, 'final_custom_cnn.h5')),
    'MobileNetV2'     : (mobilenet_model, os.path.join(OUTPUT_DIR, 'final_mobilenet.h5')),
    'EfficientNetB0'  : (effnet_model,    os.path.join(OUTPUT_DIR, 'final_efficientnet.h5')),
}

# ── Save all models in HDF5 format ────────────────────────────────────────
for model_name, (model_obj, save_path) in model_map.items():
    model_obj.save(save_path)
    print(f'Saved {model_name} → {save_path}')

# ── Save training histories ────────────────────────────────────────────────
import json
histories = {
    'custom_cnn'    : {k: [float(v) for v in vs]
                       for k, vs in cnn_history.history.items()},
    'mobilenet'     : {k: [float(v) for v in vs]
                       for k, vs in mn_history_combined.items()},
    'efficientnet'  : {k: [float(v) for v in vs]
                       for k, vs in en_history_combined.items()},
}
hist_path = os.path.join(OUTPUT_DIR, 'training_histories.json')
with open(hist_path, 'w') as f:
    json.dump(histories, f, indent=2)
print(f'Saved training histories → {hist_path}')

# ── Save evaluation results ────────────────────────────────────────────────
eval_path = os.path.join(OUTPUT_DIR, 'evaluation_results.txt')
with open(eval_path, 'w') as f:
    f.write('EMOTION RECOGNITION – EVALUATION RESULTS\n')
    f.write('='*60 + '\n\n')
    for r in all_results:
        f.write(f"Model: {r['model_name']}\n")
        f.write(f"  Test Accuracy : {r['test_acc']*100:.2f}%\n")
        f.write(f"  Macro F1      : {r['macro_f1']:.4f}\n")
        f.write(f"  Macro Prec.   : {r['macro_pre']:.4f}\n")
        f.write(f"  Macro Recall  : {r['macro_rec']:.4f}\n")
        f.write(f"\nClassification Report:\n{r['report']}\n")
        f.write('-'*60 + '\n')
print(f'Saved evaluation results → {eval_path}')

In [ ]:
# ── List all saved outputs ─────────────────────────────────────────────────
print('\n📁 Output files:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_kb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024
    print(f'  {f:45s} {size_kb:8.1f} KB')

## 10. 🏫 Classroom Engagement Adaptation Guide

This section documents how to adapt the trained emotion recognition model
for real-time student engagement detection in classrooms.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# ENGAGEMENT MAPPING
# ─────────────────────────────────────────────────────────────────────────
#
# Map the 7 FER2013 emotions to 3 engagement tiers relevant to classrooms.
# These mappings are heuristic; domain experts can refine them.
#
# ENGAGED   → happy, surprise       (positive / attentive states)
# NEUTRAL   → neutral               (baseline, neither engaged nor disengaged)
# DISENGAGED→ sad, fear, angry, disgust  (negative or off-task states)
# ─────────────────────────────────────────────────────────────────────────

ENGAGEMENT_MAP = {
    'happy'   : 'engaged',
    'surprise': 'engaged',
    'neutral' : 'neutral',
    'sad'     : 'disengaged',
    'fear'    : 'disengaged',
    'angry'   : 'disengaged',
    'disgust' : 'disengaged',
}

ENGAGEMENT_SCORE = {   # numeric score for time-series aggregation
    'engaged'   : 1.0,
    'neutral'   : 0.5,
    'disengaged': 0.0,
}

def predict_engagement(model, frame_bgr, img_size=IMG_SIZE_TL, use_rgb=True):
    """
    Predict engagement from a single BGR frame (e.g. from OpenCV).

    Parameters
    ----------
    model     : loaded Keras model
    frame_bgr : np.ndarray  – BGR image (e.g. cv2.imread output)
    img_size  : int         – model's expected input size
    use_rgb   : bool        – True for transfer-learning models, False for CNN

    Returns
    -------
    dict: emotion, confidence, engagement_level, engagement_score
    """
    # 1. Convert colour space
    if use_rgb:
        img = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    else:
        img = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)

    # 2. Resize to model input
    img = cv2.resize(img, (img_size, img_size))

    # 3. Normalise
    img = img.astype('float32') / 255.0

    # 4. Expand dims: (H, W) → (1, H, W, C)
    if not use_rgb:
        img = img[..., np.newaxis]
    img = np.expand_dims(img, axis=0)

    # 5. Predict
    probs       = model.predict(img, verbose=0)[0]
    emotion_idx = np.argmax(probs)
    emotion     = EMOTIONS[emotion_idx]
    confidence  = float(probs[emotion_idx])

    engagement  = ENGAGEMENT_MAP[emotion]
    eng_score   = ENGAGEMENT_SCORE[engagement]

    return {
        'emotion'          : emotion,
        'confidence'       : confidence,
        'all_probs'        : {e: float(p) for e, p in zip(EMOTIONS, probs)},
        'engagement_level' : engagement,
        'engagement_score' : eng_score,
    }


print('predict_engagement() function defined.')
print('\nEngagement mapping:')
for emotion, level in ENGAGEMENT_MAP.items():
    print(f'  {emotion:10s} → {level}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# REAL-TIME CLASSROOM MONITORING SKELETON
#
# Uncomment and run this cell on a machine with a webcam/camera feed.
# Uses Haar Cascade for face detection + our trained model for emotion.
# ─────────────────────────────────────────────────────────────────────────

REALTIME_CODE = '''
import cv2
import numpy as np
from collections import deque
from tensorflow import keras

# ── Load model ─────────────────────────────────────────────────────────
model = keras.models.load_model('outputs/best_mobilenet.h5')
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

# Rolling window for engagement score smoothing (5 seconds at 10 fps)
WINDOW = 50
engagement_buffer = deque(maxlen=WINDOW)

cap = cv2.VideoCapture(0)   # 0 = default camera; use RTSP URL for IP cam

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5, minSize=(48, 48))

    frame_scores = []
    for (x, y, w, h) in faces:
        face_roi = frame[y:y+h, x:x+w]
        result   = predict_engagement(model, face_roi, img_size=96, use_rgb=True)

        frame_scores.append(result['engagement_score'])

        colour = {'engaged': (0,200,0), 'neutral': (0,165,255),
                  'disengaged': (0,0,220)}[result['engagement_level']]
        cv2.rectangle(frame, (x, y), (x+w, y+h), colour, 2)
        label = f"{result['emotion']} ({result['confidence']:.0%})"
        cv2.putText(frame, label, (x, y-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, colour, 2)

    # Class-level engagement score (average across all detected faces)
    if frame_scores:
        engagement_buffer.append(np.mean(frame_scores))

    if engagement_buffer:
        avg_eng = np.mean(engagement_buffer)
        bar_txt = f"Class engagement: {avg_eng*100:.0f}%"
        cv2.putText(frame, bar_txt, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)
        # Draw engagement bar
        cv2.rectangle(frame, (10, 45), (int(300*avg_eng), 65), (0,200,0), -1)
        cv2.rectangle(frame, (10, 45), (310, 65), (200,200,200), 2)

    cv2.imshow('Classroom Engagement Monitor', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
'''

print('Real-time monitoring code ready (see REALTIME_CODE variable).')
print('To run: exec(REALTIME_CODE) — requires webcam access.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# VISUALISE ENGAGEMENT TIMELINE (simulation)
#
# Simulate a 10-minute lecture engagement trace to illustrate how the
# per-frame scores would look in a real classroom session.
# ─────────────────────────────────────────────────────────────────────────

np.random.seed(SEED)
time_minutes = np.linspace(0, 10, 300)

# Simulated engagement: high at start, dips in the middle, recovers at end
sim_engagement = (
    0.75
    + 0.15 * np.sin(np.linspace(0, 3*np.pi, 300))
    - 0.05 * (time_minutes / 10)
    + np.random.normal(0, 0.05, 300)
).clip(0, 1)

# Smooth with rolling average
from scipy.ndimage import uniform_filter1d
smoothed = uniform_filter1d(sim_engagement, size=15)

fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(time_minutes, smoothed, alpha=0.25, color='royalblue')
ax.plot(time_minutes, smoothed, color='royalblue', linewidth=2)
ax.axhline(0.65, color='green',  linestyle='--', alpha=0.6, label='Engaged threshold (0.65)')
ax.axhline(0.40, color='orange', linestyle='--', alpha=0.6, label='Alert threshold (0.40)')
ax.axhline(0.25, color='red',    linestyle='--', alpha=0.6, label='Critical threshold (0.25)')
ax.set_xlabel('Time (minutes)', fontsize=11)
ax.set_ylabel('Class Engagement Score', fontsize=11)
ax.set_title('Simulated Classroom Engagement Timeline (10-min lecture)',
             fontsize=12, fontweight='bold')
ax.set_ylim(0, 1)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'engagement_timeline_simulation.png'), dpi=150)
plt.show()

## 11. 📋 Summary, Findings & Recommendations

### Key Findings

| Topic | Finding |
|---|---|
| **Class imbalance** | `disgust` has ~16× fewer samples than `happy`. Addressed with class-weighted loss + strong augmentation. |
| **Hardest classes** | `fear` and `disgust` consistently show lowest per-class F1 due to visual similarity to `sad`/`angry`. |
| **Easiest classes** | `happy` and `surprise` are most distinctively textured and score highest F1. |
| **Overfitting** | The baseline CNN is most prone to overfitting; transfer learning models generalise better. |

### Model Recommendations for Classroom Use

| Scenario | Recommended Model | Reason |
|---|---|---|
| **Laptop / desktop** | EfficientNetB0 | Highest accuracy; runs fast on CPU for batch post-session analysis |
| **Edge / Raspberry Pi** | MobileNetV2 | Quantisation-friendly; low latency |
| **No internet / offline** | Custom CNN | No imagenet weights needed; fully self-contained |

### Future Improvement Roadmap

1. **Domain adaptation** – Fine-tune on classroom-captured face data (consider datasets like EmotiW, DAiSEE).
2. **Temporal modelling** – Feed per-frame probabilities into an LSTM/GRU for smoother engagement curves.
3. **Multi-modal fusion** – Combine facial emotion with head-pose estimation and gaze tracking.
4. **Privacy-preserving** – Apply on-device inference; never transmit raw video frames.
5. **Teacher dashboard** – Aggregate per-student scores into a heatmap overlay for real-time interventions.
6. **Confidence thresholding** – Only update the engagement score when `confidence > 0.6` to reduce noise.
7. **Model quantisation** – Apply TFLite INT8 quantisation for 4× speed-up on embedded devices.

In [ ]:
# ── Final summary printout ─────────────────────────────────────────────────
print('\n' + '='*65)
print('  FINAL RESULTS SUMMARY')
print('='*65)
for r in sorted(all_results, key=lambda x: x['test_acc'], reverse=True):
    print(f"  {r['model_name']:20s}  "
          f"Acc={r['test_acc']*100:.2f}%  "
          f"F1={r['macro_f1']:.4f}")
print('='*65)

best = max(all_results, key=lambda r: r['test_acc'])
print(f"\n🏆 Winner: {best['model_name']} with "
      f"{best['test_acc']*100:.2f}% test accuracy\n")

print('Outputs saved to ./' + OUTPUT_DIR + '/')
print('  • best_*.h5                – best checkpoints per model')
print('  • final_*.h5               – final saved models')
print('  • history_*.png            – training curves')
print('  • confusion_matrix_*.png   – confusion matrices')
print('  • model_comparison.png/csv – side-by-side comparison')
print('  • per_class_f1.png         – per-class F1 chart')
print('  • evaluation_results.txt   – full classification reports')
print('  • training_histories.json  – serialised history dicts')